# **Predictive Modeling for Text-Rich Data Using a Supervised Learning Feedback Loop with LLMs**

This project explores whether large language models can extract predictive signals from clinical trial descriptions. Using the ClinicalTrials.gov dataset, unstructured trial text is processed with Qwen 2.5 to generate structured features describing study design, eligibility constraints, and intervention complexity.

These features are combined with text embeddings and used to train a penalized logistic regression model to predict whether a clinical trial will complete successfully or terminate early.

The project also demonstrates an iterative workflow in which model feedback is used to refine LLM feature extraction.

# Dataset Overview/ Exploration

The raw dataset contains 115,480 clinical trials and 33 columns spanning trial status, dates, enrollment, sponsor details, interventions, and descriptive text fields. This mix of structured and unstructured data makes it well suited for predictive modeling with both classical ML features and LLM-derived features.

# Project Pipeline

## The workflow follows these steps:

1. LLM extracts features autonomously from trial text (Qwen 2.5 7B-Instruct)
2. Process All 1,000 Trials (**ITERATION 1**)
3. Analyze Feature Coverage (Calculating how often each feature appears across all trials.)
4. One-Hot Encode Variable Features
5. Generate Sentence Embeddings
6. Compress Embeddings with PCA
7. Combine Features
8. Supervised model (Lasso logistic regression) identifies most predictive features
9. Analyze Feature Importance (LLM features: 97.2% of importance, Embeddings: 2.8% of importance (only 5/50 used)
10. Feature importance scores- Create Feedback for LLM
11. **ITERATION 2** - Refined Prompt
12. Process Subset with Iteration 2
13. Compare Iteration Structure
14. Train Model on Iteration 2 Features
15. Compare Performance
16. Repeat to improve feature quality


In [ ]:
# STEP 1: Setup and Data Loading

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install -q transformers accelerate torch pandas numpy
!pip install -q bitsandbytes  # For 4-bit quantization if needed

# STEP 2: Load the Data

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
from tqdm import tqdm
import os

# Load your data
# You'll need to copy the file from the shared link to your Drive first
data_path = '/content/drive/MyDrive/cancer_trials_complete.csv'  # UPDATE THIS PATH
df = pd.read_csv(data_path)

# Start with a small sample for testing
df_sample = df.head(100).copy()
print(f"Loaded {len(df_sample)} trials for initial testing")
print(f"Columns: {df_sample.columns.tolist()}")
df
df_sample.to_excel('clinical_trials_sample_100.xlsx', index=False)

In [ ]:
df.columns

In [ ]:
df.info()

# Inspecting the ClinicalTrials.gov API Response

Before building the dataset pipeline, a single clinical trial record is retrieved directly from the ClinicalTrials.gov API to inspect the structure of the returned JSON. This helps identify the available study fields and understand how trial information such as study status, interventions, eligibility criteria, and outcome measures are organized.

In [ ]:
import requests

# Fetch this specific trial from the API
url = "https://clinicaltrials.gov/api/v2/studies/NCT06926972"
response = requests.get(url)
data = response.json()

# Print what the API returns
import json
print(json.dumps(data, indent=2))

In [ ]:
df["overall_status"].value_counts()

In [ ]:
success = ["COMPLETED"]
failure = ["TERMINATED", "WITHDRAWN", "SUSPENDED"]

df["target"] = df["overall_status"].apply(
    lambda x: 1 if x in success else (0 if x in failure else None)
)

df_filtered = df[df["target"].notna()].copy()
print(f"Usable trials: {len(df_filtered)}")
print(f"Success rate: {df_filtered['target'].mean():.1%}")

In [ ]:
## Step 3: Prepare Text Data for LLM Feature Extraction

# Identify key text columns for feature extraction
text_columns = [
    'brief_summary',
    'detailed_description',
    'conditions',
    'intervention_names',
    'eligibility_criteria'
]

# Check missing values in text columns
print("Missing values in text columns:")
for col in text_columns:
    missing_pct = df_filtered[col].isnull().sum() / len(df_filtered) * 100
    print(f"{col}: {missing_pct:.1f}%")

# Create combined text field for LLM
def combine_trial_text(row):
    """Combine relevant text fields into a single description"""
    parts = []

    if pd.notna(row['brief_summary']):
        parts.append(f"Summary: {row['brief_summary']}")
    if pd.notna(row['detailed_description']):
        parts.append(f"Description: {row['detailed_description']}")
    if pd.notna(row['conditions']):
        parts.append(f"Condition: {row['conditions']}")
    if pd.notna(row['intervention_names']):
        parts.append(f"Intervention: {row['intervention_names']}")
    if pd.notna(row['eligibility_criteria']):
        parts.append(f"Eligibility: {row['eligibility_criteria']}")

    return " | ".join(parts)

# Apply to dataset
df_filtered['combined_text'] = df_filtered.apply(combine_trial_text, axis=1)

# Check text length
df_filtered['text_length'] = df_filtered['combined_text'].str.len()

print("\nText statistics:")
print(df_filtered['text_length'].describe())

# Start with a smaller subset for testing
df_sample = df_filtered.sample(n=1000, random_state=42)

print(f"\nWorking with sample of {len(df_sample)} trials for initial testing")
print("Sample target distribution:")
print(df_sample['target'].value_counts())

In [ ]:
# Step 4: Setup Qwen 2.5 7B for Feature Extraction

In [ ]:
# Install required packages
!pip install -q transformers accelerate bitsandbytes sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
from tqdm import tqdm

# Check GPU
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Load Qwen 2.5 7B
model_name = "Qwen/Qwen2.5-7B-Instruct"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print("Model loaded successfully!")

# Test with one trial
test_text = df_sample.iloc[0]['combined_text'][:2000]  # First 2000 chars
print(f"\nTest trial text (truncated):\n{test_text[:500]}...")

In [ ]:
# Step 5: Design LLM Feature Extraction Prompt

In [ ]:
def create_feature_extraction_prompt(trial_text, max_chars=3000):
    """Create prompt for LLM to autonomously identify important features"""

    # Truncate text if too long
    text = trial_text[:max_chars]

    prompt = f"""You are an oncologist, analyzing a clinical trial to identify the most important characteristics that might predict whether the trial will be successfully completed or terminated/withdrawn.

Clinical Trial Information:
{text}

Analyze this trial and extract 8-12 key features that you believe are most predictive of trial success or failure. For each feature:
- Choose features that are objectively extractable from the text
- Include the feature name and its value for this specific trial
- Focus on aspects related to study design, patient population, intervention complexity, outcome measures, and feasibility

Return your analysis as a JSON object where each key is a descriptive feature name (use underscores for spaces) and each value is the extracted information for this trial.

Example format:
{{
  "feature_name_1": "value",
  "feature_name_2": "value",
  ...
}}

Return ONLY the JSON object, no additional text.
"""
    return prompt

def extract_features_with_llm(trial_text, model, tokenizer):
    """Extract features from trial text using LLM"""
    prompt = create_feature_extraction_prompt(trial_text)

    messages = [
        {"role": "system", "content": "You are a clinical trial analyst identifying predictive features."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.0,
    do_sample=False
)

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    return response

# Test on one trial
print("Testing autonomous feature extraction on sample trial...\n")
test_response = extract_features_with_llm(df_sample.iloc[0]['combined_text'], model, tokenizer)
print("LLM Response:")
print(test_response)

In [ ]:
# Now let's process all 1000 sample trials and handle the variable feature names across trials:
# This will:

# Process all 1000 trials
# Save checkpoint to your Drive
# Handle variable feature names across trials
# Track which trials succeeded

In [ ]:
import json
import time

def parse_llm_response(response):
    """Parse LLM JSON response, handling potential errors"""
    try:
        # Try to find JSON in the response
        start_idx = response.find('{')
        end_idx = response.rfind('}') + 1
        if start_idx != -1 and end_idx > start_idx:
            json_str = response[start_idx:end_idx]
            return json.loads(json_str)
        else:
            return None
    except:
        return None

def extract_features_batch(df_subset, model, tokenizer, save_path=None):
    """Extract features for a batch of trials with checkpointing"""
    results = []

    for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
        try:
            # Extract features
            response = extract_features_with_llm(row['combined_text'], model, tokenizer)
            features = parse_llm_response(response)

            if features is not None:
                features['trial_index'] = idx
                features['target'] = row['target']
                results.append(features)
            else:
                print(f"Failed to parse trial {idx}")

            # Small delay to avoid overwhelming GPU
            time.sleep(0.1)

        except Exception as e:
            print(f"Error processing trial {idx}: {e}")
            continue

    # Save checkpoint
    if save_path:
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df.to_csv(save_path, index=False)
        print(f"\nSaved checkpoint with {len(results)} trials to {save_path}")

    return pd.DataFrame(results)

# Process the 1000 sample trials
print("Extracting features from 1000 sample trials...")
print("This will take approximately 15-20 minutes...\n")

features_df = extract_features_batch(
    df_sample,
    model,
    tokenizer,
    save_path='/content/drive/MyDrive/llm_features_iteration1_sample.csv'
)

print(f"\nExtraction complete!")
print(f"Successfully extracted features from {len(features_df)} trials")
print(f"\nFeature columns found: {len(features_df.columns)}")
print(features_df.head())

In [ ]:
# Step 6: Handle Variable Feature Names (after extraction completes)
# Load the features
features_df = pd.read_csv('/content/drive/MyDrive/llm_features_iteration1_sample.csv')

print(f"Total trials: {len(features_df)}")
print(f"Total unique features: {len(features_df.columns) - 2}")  # Minus target and trial_index

# Analyze feature coverage
feature_coverage = {}
for col in features_df.columns:
    if col not in ['target', 'trial_index']:
        non_null_count = features_df[col].notna().sum()
        coverage_pct = (non_null_count / len(features_df)) * 100
        feature_coverage[col] = (non_null_count, coverage_pct)

# Sort by coverage
sorted_features = sorted(feature_coverage.items(), key=lambda x: x[1][0], reverse=True)

print(f"\nTop 30 most common features (appearing in most trials):")
for feat, (count, pct) in sorted_features[:30]:
    print(f"  {feat}: {count} trials ({pct:.1f}%)")

# Let's use features that appear in at least 5% of trials (≥49 trials)
min_coverage = 0.05
selected_features = [feat for feat, (count, pct) in feature_coverage.items()
                     if pct >= min_coverage * 100]

print(f"\n\nFeatures appearing in ≥{min_coverage*100}% of trials: {len(selected_features)}")

# Standardizing Variable LLM-Extracted Features

Because the LLM identifies features autonomously, the extracted JSON outputs contain many different feature names across trials. To make the data usable for machine learning, feature coverage is analyzed across the development sample and only features with sufficient frequency are retained.

Using a minimum coverage threshold of 5%, the feature space is reduced to 27 commonly occurring variables. This helps remove extremely sparse features while preserving the most consistently extracted trial characteristics for downstream modeling.

In [ ]:
# Step 6: Create Feature Matrix with One-Hot Encoding

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction import DictVectorizer
import scipy.sparse as sp

# Use features appearing in at least 5% of trials
min_coverage = 0.05
selected_features = [feat for feat, (count, pct) in feature_coverage.items()
                     if pct >= min_coverage * 100]

print(f"Using {len(selected_features)} features that appear in ≥{min_coverage*100}% of trials")
print(f"Selected features: {selected_features}\n")

# Create feature matrix using one-hot encoding
# Each feature value becomes a binary column

def create_feature_matrix(df, selected_features):
    """Convert LLM features to one-hot encoded matrix"""

    # Prepare data for DictVectorizer
    feature_dicts = []

    for idx, row in df.iterrows():
        feature_dict = {}
        for feat in selected_features:
            if pd.notna(row[feat]):
                # Create feature name: original_feature=value
                feature_name = f"{feat}={str(row[feat])}"
                feature_dict[feature_name] = 1
        feature_dicts.append(feature_dict)

    # Use DictVectorizer for one-hot encoding
    vectorizer = DictVectorizer(sparse=False)
    X_llm = vectorizer.fit_transform(feature_dicts)
    feature_names_llm = vectorizer.get_feature_names_out()

    print(f"Created feature matrix: {X_llm.shape}")
    print(f"  {X_llm.shape[0]} trials")
    print(f"  {X_llm.shape[1]} one-hot encoded features")

    return X_llm, feature_names_llm, vectorizer

X_llm, feature_names_llm, vectorizer = create_feature_matrix(features_df, selected_features)

# Extract target variable
y = features_df['target'].values

print(f"\nTarget variable shape: {y.shape}")
print(f"Target distribution: {np.bincount(y.astype(int))}")
print(f"Success rate: {y.mean():.2%}")

# Show some example features
print(f"\nExample one-hot encoded features (first 20):")
for i, feat in enumerate(feature_names_llm[:20]):
    print(f"  {feat}")

# Converting LLM Features into a Model Matrix

The selected LLM-extracted variables are converted into a tabular feature matrix using one-hot encoding. Each unique feature-value pair becomes a binary indicator, allowing the semantic outputs of the LLM to be represented in a format compatible with standard machine learning models.

# Feature Space Summary

After one-hot encoding, the development sample contains 984 trials and 4,424 binary LLM-derived features. This shows that a relatively small set of recurring semantic variables can still produce a high-dimensional representation because each feature can take many distinct text values across trials.

# Target Distribution

The modeling sample remains moderately imbalanced, with approximately 79% successful trials and 21% failed trials. This class distribution is preserved for downstream modeling and motivates the use of evaluation metrics beyond simple accuracy.

In [ ]:
#Step 7: Create Embeddings for Residual Information

# Install sentence-transformers
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

# Load embedding model (using a smaller, faster model)
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # 384 dimensions, fast
print("Model loaded!")

# Get the original trial texts for the 984 trials we processed
trial_indices = features_df['trial_index'].values
trial_texts = df_filtered.loc[trial_indices, 'combined_text'].values

print(f"\nGenerating embeddings for {len(trial_texts)} trials...")
print("This will take about 5-10 minutes...")

# Generate embeddings in batches
batch_size = 32
embeddings = embedding_model.encode(
    trial_texts,
    batch_size=batch_size,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Original embedding dimension: {embeddings.shape[1]}")

# Compress embeddings with PCA to reduce dimensionality
n_components = 50
pca = PCA(n_components=n_components, random_state=42)
embeddings_compressed = pca.fit_transform(embeddings)

print(f"\nCompressed embeddings shape: {embeddings_compressed.shape}")
print(f"Variance explained by {n_components} components: {pca.explained_variance_ratio_.sum():.2%}")

# Save embeddings
np.save('/content/drive/MyDrive/embeddings_iteration1.npy', embeddings_compressed)
print("\nSaved embeddings to Drive")

# Why Embeddings Are Added

The LLM-extracted features provide interpretable structured signals, while the sentence embeddings preserve broader semantic context from the raw trial text. Combining both representations allows the model to use explicit extracted concepts and residual unstructured information together.

# Output Summary

This step produces a compressed embedding matrix for the 984 processed trials, which will be combined with the one-hot encoded LLM features in the final modeling dataset.

In [ ]:
# Step 8: Combine Features and Train Penalized Regression

# Model Training

The final modeling dataset combines structured LLM-derived features with compressed text embeddings. Because the feature space is high-dimensional and sparse, a logistic regression model with L1 regularization is used to perform implicit feature selection while learning a predictive model.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# Combine LLM features + embeddings
X_combined = np.concatenate([X_llm, embeddings_compressed], axis=1)

print(f"Combined feature matrix shape: {X_combined.shape}")
print(f"  LLM features: {X_llm.shape[1]}")
print(f"  Embeddings: {embeddings_compressed.shape[1]}")
print(f"  Total: {X_combined.shape[1]}")

# Create feature names for combined matrix
embedding_feature_names = [f'embedding_{i}' for i in range(embeddings_compressed.shape[1])]
all_feature_names = list(feature_names_llm) + embedding_feature_names

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape[0]} trials")
print(f"Test set: {X_test.shape[0]} trials")

# Standardize features (important for penalized regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Lasso (L1) logistic regression for feature selection
# Start with moderate regularization
print("\n" + "="*60)
print("Training Lasso Logistic Regression (L1 penalty)")
print("="*60)

lasso_model = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=0.1,  # Regularization strength (smaller = stronger regularization)
    random_state=42,
    max_iter=1000
)

lasso_model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = lasso_model.predict(X_test_scaled)
y_pred_proba = lasso_model.predict_proba(X_test_scaled)[:, 1]

print(f"\nTest Set Performance:")
print(classification_report(y_test, y_pred, target_names=['Failure', 'Success']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.3f}")

# Cross-validation score
cv_scores = cross_val_score(lasso_model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print(f"\n5-Fold CV ROC-AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

print("\n" + "="*60)

# Model Performance

The baseline model achieves a test ROC-AUC of approximately 0.61, with cross-validated ROC-AUC of 0.59. While the model shows modest predictive signal, it struggles to identify the minority failure class effectively.

These results suggest that the LLM-extracted features capture some relevant information about trial feasibility, and further iterations with LLMs are needed.




In [ ]:
# Extract coefficients
coefficients = lasso_model.coef_[0]

# Create dataframe with feature names and coefficients
coef_df = pd.DataFrame({
    'feature': all_feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values('abs_coefficient', ascending=False)

# Filter out zero coefficients (features selected by Lasso)
selected_coef_df = coef_df[coef_df['abs_coefficient'] > 0]

print(f"Total features: {len(all_feature_names)}")
print(f"Non-zero coefficients (selected by Lasso): {len(selected_coef_df)}")
print(f"Features dropped by Lasso: {len(coef_df) - len(selected_coef_df)}")

# Top LLM features
print("\n" + "="*60)
print("TOP 20 MOST IMPORTANT LLM FEATURES")
print("="*60)
llm_features = selected_coef_df[~selected_coef_df['feature'].str.startswith('embedding_')]
for idx, row in llm_features.head(20).iterrows():
    print(f"{row['coefficient']:+.4f}  {row['feature']}")

# Embedding importance
print("\n" + "="*60)
print("EMBEDDING FEATURES ANALYSIS")
print("="*60)
embedding_features = selected_coef_df[selected_coef_df['feature'].str.startswith('embedding_')]
print(f"Embeddings with non-zero coefficients: {len(embedding_features)} out of 50")
print(f"Total embedding contribution (sum of abs coefficients): {embedding_features['abs_coefficient'].sum():.4f}")
print(f"Average LLM feature abs coefficient: {llm_features['abs_coefficient'].mean():.4f}")

if len(embedding_features) > 0:
    print(f"\nTop 10 embedding components:")
    for idx, row in embedding_features.head(10).iterrows():
        print(f"{row['coefficient']:+.4f}  {row['feature']}")

# Calculate relative importance
llm_importance = llm_features['abs_coefficient'].sum()
embedding_importance = embedding_features['abs_coefficient'].sum()
total_importance = llm_importance + embedding_importance

print("\n" + "="*60)
print("RELATIVE IMPORTANCE")
print("="*60)
print(f"LLM features total importance: {llm_importance:.4f} ({llm_importance/total_importance*100:.1f}%)")
print(f"Embeddings total importance: {embedding_importance:.4f} ({embedding_importance/total_importance*100:.1f}%)")

# Identify feature categories from selected features
feature_categories = {}
for feat in llm_features['feature'].head(30):
    category = feat.split('=')[0]  # Get the part before '='
    if category not in feature_categories:
        feature_categories[category] = 0
    feature_categories[category] += 1

print("\n" + "="*60)
print("TOP FEATURE CATEGORIES (from top 30 features)")
print("="*60)
for cat, count in sorted(feature_categories.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat}: {count} features")

# Key Findings:

### LLM features dominate (97.2%) - The structured features capture most predictive information.

### Embeddings contribute minimally (2.8%) - Only 5 out of 50 embedding dimensions are useful.

### Top categories: intervention_type, study_design, patient_population are most predictive.

### Negative coefficients dominate - Features associated with trial failure.

In [ ]:
# create feedback

In [ ]:
# Prepare feedback for LLM based on important features
top_n = 15
top_features = llm_features.head(top_n)

print("="*60)
print("FEEDBACK FOR LLM - ITERATION 2")
print("="*60)

# Extract feature categories and values that were most important
important_categories = {}
for feat_name in top_features['feature']:
    category, value = feat_name.split('=', 1)
    if category not in important_categories:
        important_categories[category] = []
    important_categories[category].append(value)

print("\nMost predictive feature categories and their important values:\n")
for category, values in important_categories.items():
    print(f"{category}:")
    for val in values[:5]:  # Show top 5 values
        print(f"  - {val}")
    print()

# Create feedback prompt for iteration 2
feedback_prompt = f"""
Based on the first iteration, these feature categories were MOST PREDICTIVE of trial success/failure:

{chr(10).join([f"- {cat}: Found to be highly important" for cat in list(important_categories.keys())[:7]])}

IMPORTANT PATTERNS DISCOVERED:
- Study design characteristics (Phase, label type) are highly predictive
- Intervention type and complexity matter significantly
- Patient population characteristics are crucial
- Study duration shows predictive power

For the NEXT iteration, please:
1. Continue extracting these important categories, but with MORE GRANULAR details
2. For 'study_design': Extract specific phase combinations, blinding details, randomization approach
3. For 'intervention_type': Be more specific about mechanism of action, combination vs single agent
4. For 'patient_population': Extract disease stage, prior treatment history, specific inclusion criteria
5. Add NEW features related to: trial logistics, site characteristics, sponsor type
6. Extract features about: enrollment criteria complexity, outcome measure difficulty, follow-up requirements

Focus on features that are:
- Objectively extractable from trial text
- Related to trial feasibility and design rigor
- Indicative of patient population complexity
"""

print("="*60)
print("FEEDBACK PROMPT FOR ITERATION 2:")
print("="*60)
print(feedback_prompt)

# Save feedback
with open('/content/drive/MyDrive/feedback_iteration1_to_2.txt', 'w') as f:
    f.write(feedback_prompt)
print("\nSaved feedback to Drive")

In [ ]:
# Step 11: LLM Feature Extraction - Iteration 2 (With Feedback)

In [ ]:
def create_feature_extraction_prompt_v2(trial_text, max_chars=3000):
    """Create improved prompt based on iteration 1 feedback"""

    text = trial_text[:max_chars]

    prompt = f"""You are analyzing a clinical trial to extract key features that predict trial success or failure.

Based on previous analysis, these categories are MOST PREDICTIVE:
- study_duration, study_design, intervention_type, patient_population
- intervention_complexity, outcome_measures, patient_population_size

Clinical Trial Information:
{text}

Extract 10-15 structured features as a JSON object. Focus on:

1. **Study Design Details**: Extract phase (I, II, III, IV, combinations), blinding (open-label, single-blind, double-blind), randomization (yes/no), multi-center status
2. **Intervention Characteristics**: Type (drug, biological, device, procedure), mechanism of action if mentioned, single vs combination therapy, administration complexity
3. **Patient Population**: Disease stage (early, advanced, metastatic, refractory), prior treatment requirements, specific inclusion criteria restrictiveness
4. **Trial Logistics**: Study duration, sample size category, number of study sites if mentioned
5. **Outcome Measures**: Primary outcome type (survival, response rate, safety, QoL), follow-up duration
6. **Feasibility Indicators**: Enrollment criteria complexity, sponsor type (industry, academic, government), special requirements

Be MORE SPECIFIC and GRANULAR than before. Extract exact details from the text.

Return ONLY a valid JSON object with descriptive feature names (use underscores). Use "not_specified" if information is not available.
"""
    return prompt

def extract_features_with_llm_v2(trial_text, model, tokenizer):
    """Extract features using improved prompt"""
    prompt = create_feature_extraction_prompt_v2(trial_text)

    messages = [
        {"role": "system", "content": "You are a clinical trial analyst extracting granular structured data."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    return response

# Test on the same first trial to see the difference
print("="*60)
print("ITERATION 2 - IMPROVED FEATURE EXTRACTION")
print("="*60)
print("\nTesting on same trial as before...\n")

test_response_v2 = extract_features_with_llm_v2(df_sample.iloc[0]['combined_text'], model, tokenizer)
print("Iteration 2 LLM Response:")
print(test_response_v2)

print("\n" + "="*60)
print("COMPARISON WITH ITERATION 1:")
print("="*60)
print("\nIteration 1 features:")
print(test_response)  # From earlier

In [ ]:
# Step 12: Process Subset with Iteration 2

#Take a random subset of 400 trials from our 984
np.random.seed(42)
subset_indices = np.random.choice(df_sample.index, size=400, replace=False)
df_subset_v2 = df_sample.loc[subset_indices]

print(f"Processing {len(df_subset_v2)} trials with Iteration 2 prompt")
print(f"Target distribution: {df_subset_v2['target'].value_counts().to_dict()}")
print(f"\nEstimated time: ~30-40 minutes")
print("Starting extraction...\n")

# Process with iteration 2
features_df_v2 = extract_features_batch(
    df_subset_v2,
    model,
    tokenizer,
    save_path='/content/drive/MyDrive/llm_features_iteration2_sample200.csv'
)

print(f"\n{'='*60}")
print(f"Iteration 2 extraction complete!")
print(f"Successfully extracted features from {len(features_df_v2)} trials")
print(f"Feature columns: {len(features_df_v2.columns)}")
print(f"{'='*60}")

# Conclusion

This project demonstrates an end-to-end pipeline for transforming unstructured clinical trial descriptions into structured features using a large language model. The extracted features can be integrated with traditional machine learning models to support predictive analysis.

The workflow illustrates how LLMs can assist in feature engineering for complex text-heavy datasets and how model feedback can guide iterative refinement of extracted variables.